In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# DATA_ROOT = "/kaggle/input/datasets/pintupatidar/signature-dataset/signature_dataset"
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     print(dirname)
    # for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import torch, subprocess, sys
print("Torch installed? ", "yes" if "torch" in sys.modules else "no")
print("CUDA available:", torch.cuda.is_available())
print("torch.version.cuda:", getattr(torch.version, "cuda", None))

# Check nvidia-smi output to see the GPU
print("\n=== nvidia-smi ===")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print("nvidia-smi not available:", e)

Torch installed?  yes
CUDA available: True
torch.version.cuda: 12.6

=== nvidia-smi ===
Mon Mar  9 11:57:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |         

In [3]:
# import torch, sys
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("torch.version.cuda:", getattr(torch.version, "cuda", None))
print("Python version:", sys.version)


Torch version: 2.9.0+cu126
CUDA available: True
torch.version.cuda: 12.6
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [5]:
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-b9tpvmux
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-b9tpvmux
  Resolved https://github.com/facebookresearch/detectron2.git to commit fd27788985af0f4ca800bca563acdb700bb890e2
  Preparing metadata (setup.py) ... done


In [6]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
torch:  2.9 ; cuda:  cu126
detectron2: 0.6


In [7]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.structures import BoxMode
from detectron2.evaluation import COCOEvaluator
from detectron2.checkpoint import DetectionCheckpointer

In [8]:


# ---------- Toggle this to switch modes ----------
FAST_SMOKE_TEST = True  # True = quick sanity run; False = real training
# -------------------------------------------------

# Path where Kaggle attaches datasets
# DATA_ROOT = "/kaggle/input/signature-dataset/signature_dataset"  # adjust if your dataset slug differs
DATA_ROOT = "/kaggle/input/datasets/pintupatidar/updated-signature-dataset/signature_dataset"

ANN_DIR = os.path.join(DATA_ROOT, "annotations")
TRAIN_IMG_DIR = os.path.join(DATA_ROOT, "train")
VAL_IMG_DIR   = os.path.join(DATA_ROOT, "valid")
TEST_IMG_DIR  = os.path.join(DATA_ROOT, "test")

TRAIN_JSON = os.path.join(ANN_DIR, "train.json")
VAL_JSON   = os.path.join(ANN_DIR, "val.json")
TEST_JSON  = os.path.join(ANN_DIR, "test.json")

assert os.path.exists(TRAIN_JSON), f"Missing: {TRAIN_JSON}"
assert os.path.exists(VAL_JSON),   f"Missing: {VAL_JSON}"
assert os.path.exists(TRAIN_IMG_DIR), f"Missing: {TRAIN_IMG_DIR}"
assert os.path.exists(VAL_IMG_DIR),   f"Missing: {VAL_IMG_DIR}"

In [9]:

for p in [TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_JSON, VAL_JSON]:
    assert os.path.exists(p), f"Path not found: {p}"


In [10]:
import os, json
from detectron2.structures import BoxMode

THING_CLASSES = ["signature"]  # single class

def load_coco_signature(json_path, img_dir):
    """
    Minimal COCO loader for a single class dataset where categories = [{id:0, name:'signature'}].
    Keeps category_id as-is (expects 0).
    """
    with open(json_path, "r") as f:
        coco = json.load(f)

    # --- Sanity checks on categories ---
    cat_ids = [c["id"] for c in coco.get("categories", [])]
    cat_names = [c["name"] for c in coco.get("categories", [])]
    assert len(cat_ids) == 1 and cat_ids[0] == 0, \
        f"Expected exactly one category with id=0, got ids={cat_ids}"
    assert cat_names[0].lower() == "signature", \
        f"Expected category name 'signature', got {cat_names[0]}"

    # Index images
    id_to_img = {img["id"]: img for img in coco["images"]}
    # Group annotations by image_id
    anns_by_img = {}
    for ann in coco["annotations"]:
        # Only accept category_id == 0
        if ann.get("category_id", None) != 0:
            # You can choose to skip or hard-fail:
            #   continue
            raise ValueError(f"Found annotation with category_id={ann.get('category_id')} "
                             f"in {json_path}. Expected only 0.")
        anns_by_img.setdefault(ann["image_id"], []).append(ann)

    # Build Detectron2 dataset dicts
    dataset_dicts = []
    missing_files = 0
    for img_id, img_info in id_to_img.items():
        file_path = os.path.join(img_dir, img_info["file_name"])
        if not os.path.exists(file_path):
            missing_files += 1
            # Skip or raise—here we skip to avoid crashing long runs
            # print(f"[WARN] Missing image file: {file_path}")
            continue

        record = {
            "file_name": file_path,
            "image_id": img_id,
            "height": img_info.get("height"),
            "width": img_info.get("width"),
            "annotations": []
        }

        for ann in anns_by_img.get(img_id, []):
            # COCO bbox is [x, y, w, h]
            bbox = ann["bbox"]
            rec_ann = {
                "bbox": bbox,
                "bbox_mode": BoxMode.XYWH_ABS,
                "category_id": 0  # single class
            }

            # If you have segmentation polygons/RLE, uncomment:
            # if "segmentation" in ann:
            #     rec_ann["segmentation"] = ann["segmentation"]

            record["annotations"].append(rec_ann)

        dataset_dicts.append(record)

    if missing_files > 0:
        print(f"[WARN] {missing_files} images referenced in {json_path} were not found in {img_dir}. Skipped.")

    return dataset_dicts

In [11]:
from detectron2.data import DatasetCatalog, MetadataCatalog

# Unregister if re-running cells
for name in ["sig_train", "sig_val"]:
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)

# Register
DatasetCatalog.register("sig_train", lambda: load_coco_signature(TRAIN_JSON, TRAIN_IMG_DIR))
DatasetCatalog.register("sig_val",   lambda: load_coco_signature(VAL_JSON,   VAL_IMG_DIR))

MetadataCatalog.get("sig_train").set(
    thing_classes=THING_CLASSES,
    json_file=TRAIN_JSON,
    image_root=TRAIN_IMG_DIR
)
MetadataCatalog.get("sig_val").set(
    thing_classes=THING_CLASSES,
    json_file=VAL_JSON,
    image_root=VAL_IMG_DIR
)

print("Registered datasets:", DatasetCatalog.list())
print("Classes (train):", MetadataCatalog.get("sig_train").thing_classes)

Registered datasets: ['coco_2014_train', 'coco_2014_val', 'coco_2014_minival', 'coco_2014_valminusminival', 'coco_2017_train', 'coco_2017_val', 'coco_2017_test', 'coco_2017_test-dev', 'coco_2017_val_100', 'keypoints_coco_2014_train', 'keypoints_coco_2014_val', 'keypoints_coco_2014_minival', 'keypoints_coco_2014_valminusminival', 'keypoints_coco_2017_train', 'keypoints_coco_2017_val', 'keypoints_coco_2017_val_100', 'coco_2017_train_panoptic_separated', 'coco_2017_train_panoptic_stuffonly', 'coco_2017_train_panoptic', 'coco_2017_val_panoptic_separated', 'coco_2017_val_panoptic_stuffonly', 'coco_2017_val_panoptic', 'coco_2017_val_100_panoptic_separated', 'coco_2017_val_100_panoptic_stuffonly', 'coco_2017_val_100_panoptic', 'lvis_v1_train', 'lvis_v1_val', 'lvis_v1_test_dev', 'lvis_v1_test_challenge', 'lvis_v0.5_train', 'lvis_v0.5_val', 'lvis_v0.5_val_rand_100', 'lvis_v0.5_test', 'lvis_v0.5_train_cocofied', 'lvis_v0.5_val_cocofied', 'cityscapes_fine_instance_seg_train', 'cityscapes_fine_sem

In [12]:
# Peek a few items to confirm category_id == 0 and paths are valid
train_samples = DatasetCatalog.get("sig_train")[:3]
for r in train_samples:
    cats = [a["category_id"] for a in r["annotations"]]
    print(os.path.basename(r["file_name"]), "->", cats[:10])

mge98e00_jpg.rf.7c96ac50cc20f8dd71b0322abe38552b.jpg -> [0]
image_16_png_jpg.rf.7d1ee7d2a97e15b635e7d731b108f83b.jpg -> [0]
image_29_jpg.rf.7bf7e8091e025f122445d3361b55d0ba.jpg -> [0]


In [13]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [16]:
import math
# --- Read train size to auto-compute schedule ---
train_len = len(DatasetCatalog.get("sig_train"))  # number of training images
IMS_PER_BATCH = 4                                  # safe on Kaggle; set 1 if OOM
iter_per_epoch = max(1, math.ceil(train_len / IMS_PER_BATCH))
EPOCHS = 30                                        # good starting point for ~3k imgs
MAX_ITER = int(EPOCHS * iter_per_epoch)


# LR milestones at ~66% and ~88% of training
step1 = int(0.66 * MAX_ITER)
step2 = int(0.88 * MAX_ITER)
WARMUP_ITERS = min(1000, int(0.05 * MAX_ITER))     # short warmup (5% or <=1000 iters)

print(f"Train images: {train_len}")
print(f"IMS_PER_BATCH: {IMS_PER_BATCH}")
print(f"Iter/epoch: {iter_per_epoch}")
print(f"MAX_ITER: {MAX_ITER}  (EPOCHS≈{EPOCHS})")
print(f"LR steps: ({step1}, {step2})  Warmup: {WARMUP_ITERS}")

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")

# Datasets
cfg.DATASETS.TRAIN = ("sig_train",)
cfg.DATASETS.TEST  = ("sig_val",)
cfg.DATALOADER.NUM_WORKERS = 2

# One class (signature)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1

# ---- Anchors tuned for signatures (often small & elongated) ----
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[8], [16], [32], [64], [128]]            # smaller-first
# cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.25], [0.5], [1.0], [2.0], [4.0]]  # allow long/wide boxes

# ---- RPN / ROI tweaks to reduce FPs on tiny marks ----
# cfg.MODEL.PROPOSAL_GENERATOR.MIN_SIZE = 10     # ignore proposals smaller than ~10px
# Keep defaults for IOU thresholds; you can adjust RPN.IOU if needed later

# Input resolution (helps small objects)
cfg.INPUT.MIN_SIZE_TRAIN = (1024,)
cfg.INPUT.MAX_SIZE_TRAIN = 1333
cfg.INPUT.MIN_SIZE_TEST  = 1024
cfg.INPUT.MAX_SIZE_TEST  = 1333

# Solver / schedule
cfg.SOLVER.IMS_PER_BATCH = IMS_PER_BATCH
# cfg.SOLVER.BASE_LR = 2.5e-4                     # a bit higher than 2.5e-4; train longer
cfg.SOLVER.BASE_LR = 0.0005                     # a bit higher than 2.5e-4; train longer
cfg.SOLVER.WARMUP_ITERS = WARMUP_ITERS
cfg.SOLVER.MAX_ITER = MAX_ITER
cfg.SOLVER.STEPS = (step1, step2)
cfg.SOLVER.GAMMA = 0.1

# Sampling: bias towards more negatives to cut FPs
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256   # more samples per image
cfg.MODEL.ROI_HEADS.POSITIVE_FRACTION = 0.25     # more negatives (defaults to 0.25)

# Mixed precision & grad clipping for stability
cfg.SOLVER.AMP.ENABLED = True
cfg.SOLVER.CLIP_GRADIENTS.ENABLED = True
cfg.SOLVER.CLIP_GRADIENTS.CLIP_TYPE = "norm"
cfg.SOLVER.CLIP_GRADIENTS.CLIP_VALUE = 1.0

# Eval/checkpoints cadence
cfg.TEST.EVAL_PERIOD = max(1000, iter_per_epoch)  # about once per epoch
cfg.SOLVER.CHECKPOINT_PERIOD = 2 * iter_per_epoch # save ~every 2 epochs

# Stricter test-time thresholds to reduce FPs when evaluating/inferencing
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5       # raise threshold
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.4         # slightly stricter NMS

# Optional: stabilize on small data by freezing early backbone stages
cfg.MODEL.BACKBONE.FREEZE_AT = 2

# Output
cfg.OUTPUT_DIR = "/kaggle/working/output_train_1"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print("Config ready. Output dir:", cfg.OUTPUT_DIR)

Train images: 2027
IMS_PER_BATCH: 4
Iter/epoch: 507
MAX_ITER: 15210  (EPOCHS≈30)
LR steps: (10038, 13384)  Warmup: 760
Config ready. Output dir: /kaggle/working/output_train_1


In [ ]:
# import detectron2.data.transforms as T
# from detectron2.data import build_detection_train_loader
# from detectron2.data import detection_utils as utils
# import torch, numpy as np, random, cv2

# def custom_mapper(dataset_dict):
#     dataset_dict = dataset_dict.copy()
#     image = utils.read_image(dataset_dict["file_name"], format="BGR")
#     annos = dataset_dict.get("annotations", [])

#     aug_list = [
#         T.ResizeShortestEdge(short_edge_length=cfg.INPUT.MIN_SIZE_TRAIN[0],
#                              max_size=cfg.INPUT.MAX_SIZE_TRAIN,
#                              sample_style="choice"),
#         T.RandomRotation(angle=[-8, 8], expand=False, sample_style="range"),
#         T.RandomBrightness(0.9, 1.1),
#         T.RandomContrast(0.9, 1.1),
#         T.RandomSaturation(0.9, 1.1),
#         T.RandomLighting(0.7),
#         T.RandomGaussianBlur(prob=0.3, sigma=(0.1, 1.5)),
#         # Random crop that sometimes yields background-only patches (hard negatives)
#         T.RandomCrop("relative_range", (0.8, 0.8)),
#         T.RandomFlip(horizontal=True, vertical=False),
#     ]
#     aug = T.AugmentationList(aug_list)
#     image, transforms = T.apply_transform_gens(aug_list, image)

#     annos = [
#         utils.transform_instance_annotations(obj, transforms, image.shape[:2])
#         for obj in annos
#         if obj.get("iscrowd", 0) == 0
#     ]

#     dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).astype("float32"))
#     instances = utils.annotations_to_instances(annos, image.shape[:2])

#     # Filter tiny boxes post-aug (to avoid noisy supervision)
#     if instances.has("gt_boxes"):
#         instances = utils.filter_empty_instances(instances, box_threshold=10.0)

#     dataset_dict["instances"] = instances
#     return dataset_dict

# # Custom trainer to plug in the mapper + evaluator
# from detectron2.engine import DefaultTrainer
# from detectron2.evaluation import COCOEvaluator

# class SigTrainer(DefaultTrainer):
#     @classmethod
#     def build_train_loader(cls, cfg):
#         return build_detection_train_loader(cfg, mapper=custom_mapper)

#     @classmethod
#     def build_evaluator(cls, cfg, dataset_name, output_folder=None):
#         if output_folder is None:
#             output_folder = os.path.join(cfg.OUTPUT_DIR, "eval")
#         os.makedirs(output_folder, exist_ok=True)
#         return COCOEvaluator(dataset_name, cfg, False, output_folder)

In [17]:
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator

class SigTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "eval")
        os.makedirs(output_folder, exist_ok=True)
        return COCOEvaluator(dataset_name, cfg, False, output_folder)

In [18]:
import warnings, logging
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("fvcore").setLevel(logging.ERROR)
logging.getLogger("detectron2").setLevel(logging.INFO)  # or ERROR to be quieter

In [ ]:
trainer = SigTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

final_pth = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
print(f"\nTraining complete. Final (if present): {final_pth}")
print(f"Eval logs: {os.path.join(cfg.OUTPUT_DIR, 'eval')}")

[03/09 12:22:43 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

model_final_280758.pkl: 167MB [00:00, 205MB/s]                             


[03/09 12:22:45 d2.engine.train_loop]: Starting training from iteration 0


W0309 12:22:48.419000 55 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


[03/09 12:22:58 d2.utils.events]:  eta: 1:59:01  iter: 19  total_loss: 0.5233  loss_cls: 0.4794  loss_box_reg: 0.01434  loss_rpn_cls: 0.01728  loss_rpn_loc: 0.01127    time: 0.4848  last_time: 0.5823  data_time: 0.0213  last_data_time: 0.0103   lr: 1.2987e-05  max_mem: 3388M


2026-03-09 12:23:01.530157: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773058981.750895      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773058981.810699      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773058982.311735      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773058982.311767      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773058982.311770      55 computation_placer.cc:177] computation placer alr

[03/09 12:23:31 d2.utils.events]:  eta: 1:59:49  iter: 39  total_loss: 0.4918  loss_cls: 0.4392  loss_box_reg: 0.01029  loss_rpn_cls: 0.02024  loss_rpn_loc: 0.008438    time: 0.4859  last_time: 0.4737  data_time: 0.0124  last_data_time: 0.0098   lr: 2.6132e-05  max_mem: 3391M
[03/09 12:23:40 d2.utils.events]:  eta: 2:00:44  iter: 59  total_loss: 0.3907  loss_cls: 0.3335  loss_box_reg: 0.01022  loss_rpn_cls: 0.01664  loss_rpn_loc: 0.009727    time: 0.4839  last_time: 0.4893  data_time: 0.0111  last_data_time: 0.0111   lr: 3.9277e-05  max_mem: 3391M
[03/09 12:23:50 d2.utils.events]:  eta: 2:01:16  iter: 79  total_loss: 0.284  loss_cls: 0.2215  loss_box_reg: 0.01885  loss_rpn_cls: 0.01566  loss_rpn_loc: 0.008971    time: 0.4865  last_time: 0.4842  data_time: 0.0109  last_data_time: 0.0101   lr: 5.2422e-05  max_mem: 3391M
[03/09 12:24:01 d2.utils.events]:  eta: 2:01:57  iter: 99  total_loss: 0.1697  loss_cls: 0.1165  loss_box_reg: 0.01089  loss_rpn_cls: 0.01385  loss_rpn_loc: 0.008687    t

In [ ]:
# import cv2
# from detectron2.engine import DefaultPredictor
# from detectron2.utils.visualizer import Visualizer, ColorMode
# import matplotlib.pyplot as plt

# # --- Load config again ---
# from detectron2.config import get_cfg
# from detectron2 import model_zoo

# cfg = get_cfg()
# cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))

# # One class
# cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1

# # Test thresholds (optional)
# cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  

# # Image size (same as training)
# # cfg.INPUT.MIN_SIZE_TEST  = 1024
# # cfg.INPUT.MAX_SIZE_TEST  = 1333

# # Load your trained model
# cfg.MODEL.WEIGHTS = "/kaggle/working/output_train_1/model_final.pth"

# predictor = DefaultPredictor(cfg)
# print("Predictor loaded!")

# # ---- Run prediction on an image ----
# # 👉 CHANGE THIS PATH to the image you want to test
# image_path = "/kaggle/input/datasets/pintupatidar/signature-dataset/signature_dataset/test/17_2_1.jpg"

# img = cv2.imread(image_path)
# outputs = predictor(img)

# # ---- Visualize ----
# v = Visualizer(img[:, :, ::-1], scale=0.8)
# out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

# result = out.get_image()[:, :, ::-1]

# save_path = "/kaggle/working/predicted_output.jpg"
# cv2.imwrite(save_path, result)

# plt.figure(figsize=(12, 12))
# plt.imshow(out.get_image()[:, :, ::-1])
# plt.axis("off")
# plt.show()

In [ ]:
# from IPython.display import FileLink
# FileLink('/kaggle/working/output_train_ver1/model_final.pth')
# # import shutil
# # shutil.make.archive('/kaggle/working/output_train_ver1','zip','/kaggle/working/')